# Imports & Data Loading

In [ ]:
!pip install nltk
import nltk
import re
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [ ]:
import numpy as np

from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer

from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Dropout

import gc
import os

In [ ]:
from tensorflow.keras.layers import SimpleRNN, LSTM, GRU, Bidirectional, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# **Data Load**

In [ ]:
#training data
train_url = 'https://drive.google.com/file/d/1FaHdT2UOnPgT7Enl-EYxrqLM8vwfBKrh/view'
id = train_url.split("/")[-2]
new_link = f'https://drive.google.com/uc?id={id}'
train_df = pd.read_csv(new_link)
display(train_df.head())

,News Headline,News Topic
0,<html> <body> News Headlines:\n <br> <b> Presi...,Business
1,<html> <body> News Headlines:\n <br> <b> Will ...,Science and Technology
2,<html> <body> News Headlines:\n <br> <b> Updat...,Business
3,<html> <body> News Headlines:\n <br> <b> Workp...,Business
4,<html> <body> News Headlines:\n <br> <b> Fish ...,Sports


# Encoding & Splitting Dataset

In [ ]:
#Label Encoding and split test and train dataset
X_train, y_train = train_df['News Headline'], train_df['News Topic']
y_train = y_train.map({'Business': 0, 'Science and Technology': 1,'Sports':2, 'World News': 3})

In [ ]:
#Split for validation

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42
)

In [ ]:
# One-hot encode the labels for the DNN, RNN models
y_train_oh = to_categorical(y_train, num_classes=4)
y_val_oh = to_categorical(y_val, num_classes=4)

# Data Preprocessing (optimum)

In [ ]:
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

In [ ]:
def optimum_preprocess(text):
    text = text.lower()

    # ✅ Remove HTML tags
    text = re.sub(r'<.*?>', '', text)

    # ✅ Remove dataset-specific phrase
    text = text.replace("news headlines:", "")

    # ✅ Remove punctuation (keep only letters)
    text = re.sub(r'[^a-z\s]', '', text)

    # ✅ Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    words = text.split()

    # ❗ Keep stopwords (important difference from optimum)

    # ✅ Lemmatization (not stemming)
    words = [lemmatizer.lemmatize(w) for w in words]

    return " ".join(words)

In [ ]:
#optimum Preprocessing for all models
X_train_optimum = X_train.apply(optimum_preprocess)
X_val_optimum = X_val.apply(optimum_preprocess)

del X_train
del X_val
del y_train
del y_val
gc.collect()

7

# Tfidf Vectorization

In [ ]:
#Tfidf Vectorization for LR model and DNN
tfidf_vectorizer = TfidfVectorizer()
X_train_tfidf_optimum = tfidf_vectorizer.fit_transform(X_train_optimum)
X_val_tfidf_optimum = tfidf_vectorizer.transform(X_val_optimum)

# Skipgram

In [ ]:
!pip install gensim
import gensim
from gensim.models import Word2Vec

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 73.5 MB/s eta 0:00:00


In [ ]:
# skipgram model training
sentences = [sentence.lower().split() for sentence in X_train_optimum]

sg_model_optimum = Word2Vec(vector_size=100, window=5, sg=1, min_count=1)

sg_model_optimum.build_vocab(sentences)
sg_model_optimum.train(sentences, total_examples=len(sentences), epochs=10)

(23165380, 28025480)

In [ ]:
# word to vector conversion
import numpy as np

def sentence_to_vectors(text):
    words = text.lower().split()
    vectors = []

    for word in words:
        if word in sg_model_optimum.wv:
            vectors.append(sg_model_optimum.wv[word])
        else:
            vectors.append(np.zeros(sg_model_optimum.vector_size))
    return np.array(vectors)


X_train_vec_optimum = [sentence_to_vectors(s) for s in X_train_optimum]
X_val_vec_optimum = [sentence_to_vectors(s) for s in X_val_optimum]

In [ ]:
# padding
from tensorflow.keras.preprocessing.sequence import pad_sequences
max_len = 50

X_train_vec_optimum = pad_sequences(X_train_vec_optimum, maxlen=max_len, padding='post', dtype='float32')
X_val_vec_optimum = pad_sequences(X_val_vec_optimum, maxlen=max_len, padding='post', dtype='float32')

In [ ]:
del X_train_optimum
del X_val_optimum

del sentences
del sg_model_optimum
gc.collect()

60

# **Model Save and  Reuse**

In [ ]:
save_path = "/content/drive/MyDrive/optimum_preprocessing"

import os
os.makedirs(save_path, exist_ok=True)

from scipy import sparse
import numpy as np

# TF-IDF
sparse.save_npz(f"{save_path}/X_train_tfidf_optimum.npz", X_train_tfidf_optimum)
sparse.save_npz(f"{save_path}/X_val_tfidf_optimum.npz", X_val_tfidf_optimum)

# Skipgram
np.save(f"{save_path}/X_train_vec_optimum.npy", X_train_vec_optimum)
np.save(f"{save_path}/X_val_vec_optimum.npy", X_val_vec_optimum)

# Labels
np.save(f"{save_path}/y_train_oh.npy", y_train_oh)
np.save(f"{save_path}/y_val_oh.npy", y_val_oh)

print("✅ Saved to Google Drive!")

✅ Saved to Google Drive!


In [ ]:
save_path = "/content/drive/MyDrive/optimum_preprocessing"

from scipy import sparse
import numpy as np

X_train_tfidf_optimum = sparse.load_npz(f"{save_path}/X_train_tfidf_optimum.npz")
X_val_tfidf_optimum   = sparse.load_npz(f"{save_path}/X_val_tfidf_optimum.npz")

X_train_vec_optimum = np.load(f"{save_path}/X_train_vec_optimum.npy")
X_val_vec_optimum   = np.load(f"{save_path}/X_val_vec_optimum.npy")

y_train_oh = np.load(f"{save_path}/y_train_oh.npy")
y_val_oh   = np.load(f"{save_path}/y_val_oh.npy")

In [ ]:
max_len = X_train_vec_optimum.shape[1]
embedding_dim = X_train_vec_optimum.shape[2]

# DNN

In [ ]:
def dnn_optimum(units=128, dropout=0.5, dense_units=64, lr=0.001):
    dnn_model_optimum = Sequential([
        Dense(units, activation="relu", input_shape=(X_train_tfidf_optimum.shape[1],)),
        Dropout(dropout),
        Dense(dense_units, activation="relu"),
        Dropout(dropout),
        Dense(32, activation="relu"),
        Dense(4, activation="softmax")
    ])
    dnn_model_optimum.compile(
        optimizer=Adam(learning_rate=lr),
        loss="categorical_crossentropy",
        metrics=["accuracy"])
    return dnn_model_optimum

# SimpleRNN

In [ ]:
def simple_rnn_optimum(units=128, dropout=0.5, dense_units=64, lr=0.001):

    simple_rnn_model_optimum = Sequential([
        SimpleRNN(units, input_shape=(max_len, embedding_dim)),
        Dropout(dropout),
        Dense(dense_units, activation='relu'),
        Dense(4, activation='softmax')
    ])

    simple_rnn_model_optimum.compile(
        optimizer=Adam(learning_rate=lr),
        loss='categorical_crossentropy',
        metrics=['accuracy'],
    )
    return simple_rnn_model_optimum

# LSTM

In [ ]:
def lstm_optimum(units=128, dropout=0.5, dense_units=64, lr=0.001):

    lstm_model_optimum = Sequential([
        LSTM(units, input_shape=(max_len, embedding_dim)),
        Dropout(dropout),
        Dense(dense_units, activation='relu'),
        Dense(4, activation='softmax')
    ])

    lstm_model_optimum.compile(
        optimizer=Adam(learning_rate=lr),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return lstm_model_optimum

# GRU

In [ ]:
def gru_optimum(units=128, dropout=0.5, dense_units=64, lr=0.001):

    gru_model_optimum = Sequential([
        GRU(units, input_shape=(max_len, embedding_dim)),
        Dropout(dropout),
        Dense(dense_units, activation='relu'),
        Dense(4, activation='softmax')
    ])

    gru_model_optimum.compile(
        optimizer=Adam(learning_rate=lr),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return gru_model_optimum

# Bidirectional SimpleRNN

In [ ]:
def bidirectional_simple_rnn_optimum(units=128, dropout=0.5, dense_units=64, lr=0.001):

    bidirectional_simple_rnn_model_optimum = Sequential([
        Input(shape=(max_len, embedding_dim)), # Explicit Input layer
        Bidirectional(SimpleRNN(units, input_shape=(max_len, embedding_dim))),
        Dropout(dropout),
        Dense(dense_units, activation='relu'),
        Dense(4, activation='softmax')
    ])

    bidirectional_simple_rnn_model_optimum.compile(
        optimizer=Adam(learning_rate=lr),
        loss='categorical_crossentropy',
        metrics=['accuracy'],
    )

    return bidirectional_simple_rnn_model_optimum

# Bidirectional LSTM

In [ ]:
def bidirectional_lstm_optimum(units=128, dropout=0.5, dense_units=64, lr=0.001):

    bidirectional_lstm_model_optimum = Sequential([
        Input(shape=(max_len, embedding_dim)), # Explicit Input layer
        Bidirectional(LSTM(units)),
        Dropout(dropout),
        Dense(dense_units, activation='relu'),
        Dense(4, activation='softmax')
    ])

    bidirectional_lstm_model_optimum.compile(
        optimizer=Adam(learning_rate=lr),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return bidirectional_lstm_model_optimum

# Bidirectional GRU

In [ ]:
def bidirectional_gru_optimum(units=128, dropout=0.5, dense_units=64, lr=0.001):

    bidirectional_gru_model_optimum = Sequential([
        Input(shape=(max_len, embedding_dim)), # Explicit Input layer
        Bidirectional(GRU(units)),
        Dropout(dropout),
        Dense(dense_units, activation='relu'),
        Dense(4, activation='softmax')
    ])

    bidirectional_gru_model_optimum.compile(
        optimizer=Adam(learning_rate=lr),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return bidirectional_gru_model_optimum

# **Tuning Part**

In [ ]:
from sklearn.metrics import accuracy_score, f1_score
import numpy as np

def evaluate(model, X, y_oh):
    y_pred = np.argmax(model.predict(X, verbose=0), axis=1)
    y_true = np.argmax(y_oh, axis=1)

    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='macro')

    return acc, f1

In [ ]:
def run_experiment(model_fn, config, X_train, y_train, X_val, y_val, branch, model_name, run_id):

    model = model_fn(
        units=config["units"],
        dropout=config["dropout"],
        dense_units=config["dense_units"],
        lr=config["lr"]
    )

    early_stop = EarlyStopping(
        monitor='val_loss',
        patience=2,
        restore_best_weights=True
    )

    model.fit(
        X_train, y_train,
        epochs=config["epochs"],
        batch_size=config["batch_size"],
        validation_data=(X_val, y_val),
        callbacks=[early_stop],
        verbose=1
    )

    val_acc, val_f1 = evaluate(model, X_val, y_val)
    train_acc, train_f1 = evaluate(model, X_train, y_train)

    res =  {
        "run_id": run_id,
        "branch": branch,
        "model": model_name,
        "units": config["units"],
        "dropout": config["dropout"],
        "dense_units": config["dense_units"],
        "lr": config["lr"],
        "batch_size": config["batch_size"],
        "epochs": config["epochs"],
        "train_accuracy": train_acc,
        "train_f1": train_f1,
        "val_accuracy": val_acc,
        "val_f1": val_f1
    }

    del model
    tf.keras.backend.clear_session()
    gc.collect()

    return res

In [ ]:
models = [
    {
        "name": "DNN",
        "fn": dnn_optimum,
        "X_train": X_train_tfidf_optimum,
        "X_val": X_val_tfidf_optimum
    },
    {
        "name": "SimpleRNN",
        "fn": simple_rnn_optimum,
        "X_train": X_train_vec_optimum,
        "X_val": X_val_vec_optimum
    },
    {
        "name": "LSTM",
        "fn": lstm_optimum,
        "X_train": X_train_vec_optimum,
        "X_val": X_val_vec_optimum
    },
    {
        "name": "GRU",
        "fn": gru_optimum,
        "X_train": X_train_vec_optimum,
        "X_val": X_val_vec_optimum
    },
    {
        "name": "Bi_SimpleRNN",
        "fn": bidirectional_simple_rnn_optimum,
        "X_train": X_train_vec_optimum,
        "X_val": X_val_vec_optimum
    },
    {
        "name": "Bi_LSTM",
        "fn": bidirectional_lstm_optimum,
        "X_train": X_train_vec_optimum,
        "X_val": X_val_vec_optimum
    },
    {
        "name": "Bi_GRU",
        "fn": bidirectional_gru_optimum,
        "X_train": X_train_vec_optimum,
        "X_val": X_val_vec_optimum
    }
]

In [ ]:
configs = [
    {"units": 64, "dropout": 0.30, "dense_units": 64, "lr": 0.001, "batch_size": 64, "epochs": 6},
    {"units": 128,"dropout": 0.50,"dense_units": 64,"lr": 0.001,"batch_size": 32,"epochs": 8},
    {"units": 128,"dropout": 0.20,"dense_units": 128,"lr": 0.001,"batch_size": 32,"epochs": 10},
    {"units": 32,"dropout": 0.60,"dense_units": 32,"lr": 0.0005,"batch_size": 128,"epochs": 4},
    {"units": 64,"dropout": 0.40,"dense_units": 128,"lr": 0.005,"batch_size": 64,"epochs": 5}
]

model_configs = {
    "DNN": [
        {"units": 32, "dropout": 0.55, "dense_units": 32, "lr": 0.0005, "batch_size": 128, "epochs": 5},
        {"units": 48, "dropout": 0.50, "dense_units": 64, "lr": 0.0005, "batch_size": 128, "epochs": 5},
        {"units": 64, "dropout": 0.45, "dense_units": 64, "lr": 0.0010, "batch_size": 64,  "epochs": 6},
    ],

    "SimpleRNN": [
        {"units": 48, "dropout": 0.35, "dense_units": 64, "lr": 0.0010, "batch_size": 64,  "epochs": 6},
        {"units": 64, "dropout": 0.30, "dense_units": 64, "lr": 0.0010, "batch_size": 64,  "epochs": 6},
        {"units": 96, "dropout": 0.40, "dense_units": 32, "lr": 0.0005, "batch_size": 128, "epochs": 5},
    ],

    "LSTM": [
        {"units": 96,  "dropout": 0.20, "dense_units": 128, "lr": 0.0010, "batch_size": 32, "epochs": 10},
        {"units": 128, "dropout": 0.20, "dense_units": 128, "lr": 0.0010, "batch_size": 32, "epochs": 10},
        {"units": 128, "dropout": 0.25, "dense_units": 64,  "lr": 0.0005, "batch_size": 32, "epochs": 8},
    ],

    "GRU": [
        {"units": 96,  "dropout": 0.20, "dense_units": 128, "lr": 0.0010, "batch_size": 32, "epochs": 10},
        {"units": 128, "dropout": 0.20, "dense_units": 128, "lr": 0.0010, "batch_size": 32, "epochs": 10},
        {"units": 128, "dropout": 0.25, "dense_units": 64,  "lr": 0.0005, "batch_size": 32, "epochs": 8},
    ],

    "Bi_SimpleRNN": [
        {"units": 48, "dropout": 0.35, "dense_units": 64, "lr": 0.0010, "batch_size": 64,  "epochs": 6},
        {"units": 64, "dropout": 0.30, "dense_units": 64, "lr": 0.0010, "batch_size": 64,  "epochs": 6},
        {"units": 96, "dropout": 0.40, "dense_units": 32, "lr": 0.0005, "batch_size": 128, "epochs": 5},
    ],

    "Bi_LSTM": [
        {"units": 96,  "dropout": 0.20, "dense_units": 128, "lr": 0.0010, "batch_size": 32, "epochs": 10},
        {"units": 128, "dropout": 0.20, "dense_units": 128, "lr": 0.0010, "batch_size": 32, "epochs": 10},
        {"units": 128, "dropout": 0.25, "dense_units": 64,  "lr": 0.0005, "batch_size": 32, "epochs": 8},
    ],

    "Bi_GRU": [
        {"units": 96,  "dropout": 0.20, "dense_units": 128, "lr": 0.0010, "batch_size": 32, "epochs": 10},
        {"units": 128, "dropout": 0.20, "dense_units": 128, "lr": 0.0010, "batch_size": 32, "epochs": 10},
        {"units": 128, "dropout": 0.25, "dense_units": 64,  "lr": 0.0005, "batch_size": 32, "epochs": 8},
    ],
}

In [ ]:
run_id = 48

In [ ]:
file_path = "/content/drive/MyDrive/optimum_preprocessing/optimum_tuning_results.csv"
for m in range(4, 7):
    model_info = models[m]
    model_name = model_info["name"]
    print(f"\n🔹 Tuning {model_name}...")

    for i in range(3):
        cfg = model_configs[model_name][i]

        res = run_experiment(
            model_fn=model_info["fn"],
            config=cfg,
            X_train=model_info["X_train"],
            y_train=y_train_oh,
            X_val=model_info["X_val"],
            y_val=y_val_oh,
            branch="Optimum",
            model_name = model_info["name"],
            run_id=run_id
        )

        new_df = pd.DataFrame([res])
        new_df.to_csv(
            file_path,
            mode='a',
            header=not os.path.exists(file_path),
            index=False
        )

        print(f"Saved run {run_id}")

        run_id += 1


🔹 Tuning Bi_SimpleRNN...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/6
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 40s 29ms/step - accuracy: 0.8503 - loss: 0.4349 - val_accuracy: 0.8932 - val_loss: 0.3284
Epoch 2/6
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 34s 28ms/step - accuracy: 0.8850 - loss: 0.3482 - val_accuracy: 0.8786 - val_loss: 0.3534
Epoch 3/6
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 34s 29ms/step - accuracy: 0.8870 - loss: 0.3418 - val_accuracy: 0.8875 - val_loss: 0.3423
Saved run 48


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/6
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 41s 30ms/step - accuracy: 0.8602 - loss: 0.4085 - val_accuracy: 0.8800 - val_loss: 0.3645
Epoch 2/6
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 38s 32ms/step - accuracy: 0.8867 - loss: 0.3449 - val_accuracy: 0.8773 - val_loss: 0.3640
Epoch 3/6
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 39s 33ms/step - accuracy: 0.8905 - loss: 0.3308 - val_accuracy: 0.8982 - val_loss: 0.3157
Epoch 4/6
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 37s 31ms/step - accuracy: 0.8777 - loss: 0.3743 - val_accuracy: 0.8841 - val_loss: 0.3675
Epoch 5/6
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 40s 30ms/step - accuracy: 0.8889 - loss: 0.3362 - val_accuracy: 0.8961 - val_loss: 0.3095
Epoch 6/6
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 43s 36ms/step - accuracy: 0.8932 - loss: 0.3241 - val_accuracy: 0.8927 - val_loss: 0.3301
Saved run 49


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Epoch 1/5
597/597 ━━━━━━━━━━━━━━━━━━━━ 41s 62ms/step - accuracy: 0.8411 - loss: 0.4525 - val_accuracy: 0.8964 - val_loss: 0.3199
Epoch 2/5
597/597 ━━━━━━━━━━━━━━━━━━━━ 36s 60ms/step - accuracy: 0.8921 - loss: 0.3272 - val_accuracy: 0.8937 - val_loss: 0.3123
Epoch 3/5
597/597 ━━━━━━━━━━━━━━━━━━━━ 42s 70ms/step - accuracy: 0.8958 - loss: 0.3146 - val_accuracy: 0.8955 - val_loss: 0.3124
Epoch 4/5
597/597 ━━━━━━━━━━━━━━━━━━━━ 76s 61ms/step - accuracy: 0.8978 - loss: 0.3025 - val_accuracy: 0.8922 - val_loss: 0.3095
Epoch 5/5
597/597 ━━━━━━━━━━━━━━━━━━━━ 42s 62ms/step - accuracy: 0.9004 - loss: 0.2923 - val_accuracy: 0.8967 - val_loss: 0.3026
Saved run 50

🔹 Tuning Bi_LSTM...
Epoch 1/10
2386/2386 ━━━━━━━━━━━━━━━━━━━━ 189s 78ms/step - accuracy: 0.8938 - loss: 0.3082 - val_accuracy: 0.9081 - val_loss: 0.2664
Epoch 2/10
2386/2386 ━━━━━━━━━━━━━━━━━━━━ 206s 80ms/step - accuracy: 0.9121 - loss: 0.2493 - val_accuracy: 0.9157 - val_loss: 0.2405
Epoch 3/10
2386/2386 ━━━━━━━━━━━━━━━━━━━━ 197s 82ms/ste